# Part 2 — Sufficiency → Representation → Recoverability
## Frozen pretrained Transformers: Qwen2.5 0.5B and 1.5B

**Primary question:** once the input becomes information-theoretically sufficient at \(L=k\), is the known Markov sufficient state recoverable from internal pretrained representations?

Strict design: frozen LLMs, random token-ID remapping per seed, separate train/val/test samples, selected layers, linear probes at every layer, final-layer MLP control, theoretical \(H^*\) and \(H_L\), DONE markers, clean exports.

All outputs are written to `Markov_Context_Paper/Part2/`.


In [ ]:

# 0. Package preflight
import sys, subprocess, importlib.util
req={"transformers":"transformers>=4.48,<5","accelerate":"accelerate>=1.0","safetensors":"safetensors>=0.4"}
missing=[p for p in req if importlib.util.find_spec(p) is None]
if missing:
    print("Installing:", missing)
    subprocess.check_call([sys.executable,"-m","pip","install","-q",*[req[p] for p in missing]])
print("Package preflight complete")


In [ ]:

# 1. Imports / GPU / folders
import os,json,math,time,random,hashlib,gc,zipfile,platform
from dataclasses import dataclass,asdict
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import TensorDataset,DataLoader
from transformers import AutoModelForCausalLM
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_BF16=bool(torch.cuda.is_available() and torch.cuda.is_bf16_supported())
MODEL_DTYPE=torch.bfloat16 if USE_BF16 else torch.float16
PROJECT_ROOT=Path.cwd()/"Markov_Context_Paper"/"Part2"
RUNS_DIR=PROJECT_ROOT/"runs"; EXPORTS_DIR=PROJECT_ROOT/"exports"; CSV_DIR=EXPORTS_DIR/"csv"; FIG_DIR=EXPORTS_DIR/"figures"; CACHE_DIR=PROJECT_ROOT/"hf_cache"; THEORY_DIR=PROJECT_ROOT/"theory_cache"
for p in [PROJECT_ROOT,RUNS_DIR,EXPORTS_DIR,CSV_DIR,FIG_DIR,CACHE_DIR,THEORY_DIR]: p.mkdir(parents=True,exist_ok=True)
os.environ["HF_HOME"]=str(CACHE_DIR)
print("Torch",torch.__version__,"GPU",torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,"dtype",MODEL_DTYPE)
print("PROJECT_ROOT",PROJECT_ROOT)


In [ ]:

# 2. Strict experiment configuration
MODELS={"qwen05":"Qwen/Qwen2.5-0.5B","qwen15":"Qwen/Qwen2.5-1.5B"}
V=8; KS=[2,3,4]; SEEDS=list(range(10))
def context_lengths(k): return [max(1,k-1),k,k+1]
N_TRAIN=12000; N_VAL=3000; N_TEST=6000
EXTRACT_BATCH={"qwen05":256,"qwen15":128}
N_SELECTED_LAYERS=5
PROBE_BATCH=512; LINEAR_EPOCHS=25; MLP_EPOCHS=30; PATIENCE=4
LINEAR_LR=1e-3; MLP_LR=5e-4; PROBE_WD=1e-4
RUN_FINAL_MLP_CONTROL=True
SMOKE_TEST= False
if SMOKE_TEST:
    MODELS={"qwen05":"Qwen/Qwen2.5-0.5B"}; KS=[2]; SEEDS=[0]
    N_TRAIN,N_VAL,N_TEST=512,256,256; LINEAR_EPOCHS=3; MLP_EPOCHS=3
print(MODELS,KS,SEEDS,N_TRAIN,N_VAL,N_TEST)


In [ ]:

# 3. Reproducibility helpers
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
def safe_json(o):
    if isinstance(o,Path): return str(o)
    if isinstance(o,np.integer): return int(o)
    if isinstance(o,np.floating): return float(o)
    if isinstance(o,torch.dtype): return str(o)
    return o
def save_json(p,o):
    p.parent.mkdir(parents=True,exist_ok=True)
    with open(p,"w",encoding="utf-8") as f: json.dump(o,f,indent=2,default=safe_json)
def stable_hash(d,n=12): return hashlib.sha256(json.dumps(d,sort_keys=True,default=safe_json).encode()).hexdigest()[:n]
def clean_gpu():
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
save_json(PROJECT_ROOT/"environment.json",{"python":sys.version,"torch":torch.__version__,"cuda":torch.version.cuda,"gpu":torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,"dtype":str(MODEL_DTYPE),"models":MODELS})


## Markov data and information-theoretic floor
For each sample we retain the full \(k\)-state, the observed context of length \(L\), and the next token.  \(H_L\) is estimated directly from the known synthetic kernel.

In [ ]:

def context_to_index(c,V):
    idx=torch.zeros(c.shape[:-1],dtype=torch.long,device=c.device)
    for j in range(c.shape[-1]): idx=idx*V+c[...,j]
    return idx
def generate_kernel(V,k,seed,device=DEVICE):
    g=torch.Generator(device=device); g.manual_seed(seed+10000019)
    return torch.softmax(torch.randn((V**k,V),generator=g,device=device),dim=-1)
@torch.no_grad()
def generate_dataset(P,V,k,L,n,seed,batch_size=1024):
    g=torch.Generator(device=DEVICE); g.manual_seed(seed+7000009)
    B=min(batch_size,n); hlen=max(k,L)
    hist=torch.randint(0,V,(B,hlen),generator=g,device=DEVICE)
    for _ in range(256):
        probs=P[context_to_index(hist[:,-k:],V)]; y=torch.multinomial(probs,1,generator=g).squeeze(1); hist=torch.cat([hist[:,1:],y[:,None]],1)
    states=[]; obs=[]; ys=[]; rem=n
    while rem>0:
        take=min(B,rem); h=hist[:take]
        st=h[:,-k:].clone(); ob=h[:,-L:].clone(); probs=P[context_to_index(st,V)]; y=torch.multinomial(probs,1,generator=g).squeeze(1)
        states.append(st.cpu()); obs.append(ob.cpu()); ys.append(y.cpu())
        probs_all=P[context_to_index(hist[:,-k:],V)]; ya=torch.multinomial(probs_all,1,generator=g).squeeze(1); hist=torch.cat([hist[:,1:],ya[:,None]],1)
        rem-=take
    return torch.cat(states).long(),torch.cat(obs).long(),torch.cat(ys).long()
@torch.no_grad()
def estimate_H_star(P):
    p=P.double(); return float((-(p*torch.log(p.clamp_min(1e-30))).sum(-1)).mean())
@torch.no_grad()
def estimate_H_L(P,V,k,L,n=300000,seed=999):
    if L>=k: return estimate_H_star(P)
    st,ob,_=generate_dataset(P,V,k,L,n,seed,4096); st=st.to(DEVICE); ob=ob.to(DEVICE)
    pf=P[context_to_index(st,V)].double(); oi=context_to_index(ob,V); nc=V**L
    sm=torch.zeros((nc,V),dtype=torch.float64,device=DEVICE); ct=torch.zeros(nc,dtype=torch.float64,device=DEVICE)
    sm.index_add_(0,oi,pf); ct.index_add_(0,oi,torch.ones(len(oi),dtype=torch.float64,device=DEVICE))
    q=sm/ct[:,None].clamp_min(1.0); qb=q[oi].clamp_min(1e-30)
    return float((-(pf*torch.log(qb)).sum(-1)).mean())


## Random token-ID mapping control
Markov symbols are mapped directly to randomly selected ordinary vocabulary IDs. The mapping is randomized by seed and saved, reducing dependence on hand-picked lexical tokens.

In [ ]:

def choose_token_ids(model,V,seed):
    vs=model.config.vocab_size; special=set()
    for a in ["bos_token_id","eos_token_id","pad_token_id"]:
        v=getattr(model.config,a,None)
        if v is not None: special.update(v if isinstance(v,list) else [int(v)])
    lo=min(1000,max(0,vs//20)); hi=max(lo+V+100,vs-1000)
    cand=np.array([i for i in range(lo,hi) if i not in special],dtype=np.int64)
    rng=np.random.default_rng(seed+91337); return torch.tensor(rng.choice(cand,V,replace=False),dtype=torch.long)
def map_symbols(x,mapping): return mapping[x]


In [ ]:

# 6. Frozen model + selected layers
def load_frozen(model_id):
    print("Loading",model_id)
    m=AutoModelForCausalLM.from_pretrained(model_id,cache_dir=str(CACHE_DIR),torch_dtype=MODEL_DTYPE,low_cpu_mem_usage=True).to(DEVICE).eval()
    for p in m.parameters(): p.requires_grad_(False)
    return m
def layer_indices(model):
    n=int(model.config.num_hidden_layers); idx=sorted(set(int(round(x)) for x in np.linspace(0,n,N_SELECTED_LAYERS)))
    if idx[-1]!=n: idx.append(n)
    return idx
@torch.no_grad()
def extract(model,ids,layers,batch_size):
    reps={li:[] for li in layers}
    for s in range(0,len(ids),batch_size):
        b=ids[s:s+batch_size].to(DEVICE)
        out=model(input_ids=b,attention_mask=torch.ones_like(b),output_hidden_states=True,use_cache=False,return_dict=True)
        for li in layers: reps[li].append(out.hidden_states[li][:,-1,:].detach().cpu().to(torch.float16))
        del out,b; clean_gpu()
    return {li:torch.cat(v) for li,v in reps.items()}


## Probes
Primary: factorized full-state recovery. Secondary: next-token prediction. Linear probes are used at every selected layer; an MLP is used at the final layer as a nonlinear decoding control.

In [ ]:

class LinearState(nn.Module):
    def __init__(self,d,k,V): super().__init__(); self.k=k; self.V=V; self.h=nn.Linear(d,k*V)
    def forward(self,x): return self.h(x).view(-1,self.k,self.V)
class MLPState(nn.Module):
    def __init__(self,d,k,V):
        super().__init__(); h=min(1024,max(256,d)); self.k=k; self.V=V; self.net=nn.Sequential(nn.Linear(d,h),nn.GELU(),nn.Linear(h,k*V))
    def forward(self,x): return self.net(x).view(-1,self.k,self.V)
class LinearNext(nn.Module):
    def __init__(self,d,V): super().__init__(); self.h=nn.Linear(d,V)
    def forward(self,x): return self.h(x)
class MLPNext(nn.Module):
    def __init__(self,d,V): super().__init__(); h=min(1024,max(256,d)); self.net=nn.Sequential(nn.Linear(d,h),nn.GELU(),nn.Linear(h,V))
    def forward(self,x): return self.net(x)
def sloss(logits,y): return F.cross_entropy(logits.reshape(-1,logits.shape[-1]),y.reshape(-1))
def train_probe(model,Xtr,ytr,Xva,yva,task,epochs,lr):
    model=model.to(DEVICE); opt=torch.optim.AdamW(model.parameters(),lr=lr,weight_decay=PROBE_WD)
    tr=DataLoader(TensorDataset(Xtr.float(),ytr),batch_size=PROBE_BATCH,shuffle=True); va=DataLoader(TensorDataset(Xva.float(),yva),batch_size=PROBE_BATCH)
    best=float('inf'); best_state=None; bad=0
    for ep in range(epochs):
        model.train()
        for xb,yb in tr:
            xb,yb=xb.to(DEVICE),yb.to(DEVICE); opt.zero_grad(set_to_none=True); z=model(xb); loss=sloss(z,yb) if task=='state' else F.cross_entropy(z,yb); loss.backward(); opt.step()
        model.eval(); tot=0.0
        with torch.no_grad():
            for xb,yb in va:
                xb,yb=xb.to(DEVICE),yb.to(DEVICE); z=model(xb); loss=sloss(z,yb) if task=='state' else F.cross_entropy(z,yb); tot+=loss.item()*len(xb)
        v=tot/len(Xva)
        if v<best-1e-5: best=v; best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}; bad=0
        else:
            bad+=1
            if bad>=PATIENCE: break
    if best_state: model.load_state_dict(best_state)
    return model
@torch.no_grad()
def eval_state(model,X,y):
    model.eval(); dl=DataLoader(TensorDataset(X.float(),y),batch_size=PROBE_BATCH); loss=0; n=0; pc=None; exact=0
    for xb,yb in dl:
        xb,yb=xb.to(DEVICE),yb.to(DEVICE); z=model(xb); l=sloss(z,yb); p=z.argmax(-1); b=(p==yb).sum(0).cpu(); pc=b if pc is None else pc+b; exact+=(p==yb).all(1).sum().item(); loss+=l.item()*len(xb); n+=len(xb)
    pa=(pc.float()/n).numpy(); return dict(state_ce=loss/n,state_mean_pos_acc=float(pa.mean()),state_oldest_acc=float(pa[0]),state_last_acc=float(pa[-1]),state_exact_acc=float(exact/n),state_pos_acc=json.dumps(pa.tolist()))
@torch.no_grad()
def eval_next(model,X,y):
    model.eval(); dl=DataLoader(TensorDataset(X.float(),y),batch_size=PROBE_BATCH); loss=0; cor=0; n=0
    for xb,yb in dl:
        xb,yb=xb.to(DEVICE),yb.to(DEVICE); z=model(xb); l=F.cross_entropy(z,yb); loss+=l.item()*len(xb); cor+=(z.argmax(-1)==yb).sum().item(); n+=len(xb)
    return dict(next_ce=loss/n,next_acc=cor/n)


In [ ]:

# 8. One configuration
@dataclass
class ExpConfig:
    model_key:str; model_id:str; k:int; L:int; seed:int
    def run_id(self): return f"{self.model_key}_k{self.k}_L{self.L}_seed{self.seed}_{stable_hash(asdict(self))}"
def exp_dir(c): return RUNS_DIR/c.run_id()
def run_config(model,c):
    rd=exp_dir(c); rd.mkdir(parents=True,exist_ok=True); done=rd/"DONE"; outcsv=rd/"probe_results.csv"
    if done.exists() and outcsv.exists(): print("SKIP DONE",c.run_id()); return pd.read_csv(outcsv)
    set_seed(c.seed); P=generate_kernel(V,c.k,c.seed); Hs=estimate_H_star(P)
    tf=THEORY_DIR/f"V{V}_k{c.k}_L{c.L}_seed{c.seed}.json"
    if tf.exists(): HL=float(json.load(open(tf))["H_L"])
    else: HL=estimate_H_L(P,V,c.k,c.L,300000,c.seed+31337); save_json(tf,{"H_star":Hs,"H_L":HL,"info_gap":HL-Hs})
    mp=PROJECT_ROOT/"token_mappings"/f"{c.model_key}_seed{c.seed}.json"; mp.parent.mkdir(exist_ok=True)
    if mp.exists(): mapping=torch.tensor(json.load(open(mp))["token_ids"],dtype=torch.long)
    else:
        ms=c.seed+int(hashlib.sha256(c.model_key.encode()).hexdigest()[:6],16); mapping=choose_token_ids(model,V,ms); save_json(mp,{"token_ids":mapping.tolist(),"seed":ms})
    tr_s,tr_o,tr_y=generate_dataset(P,V,c.k,c.L,N_TRAIN,c.seed+101); va_s,va_o,va_y=generate_dataset(P,V,c.k,c.L,N_VAL,c.seed+202); te_s,te_o,te_y=generate_dataset(P,V,c.k,c.L,N_TEST,c.seed+303)
    layers=layer_indices(model); tr=extract(model,map_symbols(tr_o,mapping),layers,EXTRACT_BATCH[c.model_key]); va=extract(model,map_symbols(va_o,mapping),layers,EXTRACT_BATCH[c.model_key]); te=extract(model,map_symbols(te_o,mapping),layers,EXTRACT_BATCH[c.model_key])
    rows=[]
    for li in layers:
        Xtr,Xva,Xte=tr[li],va[li],te[li]; d=Xtr.shape[1]
        sp=train_probe(LinearState(d,c.k,V),Xtr,tr_s,Xva,va_s,'state',LINEAR_EPOCHS,LINEAR_LR); sm=eval_state(sp,Xte,te_s)
        npb=train_probe(LinearNext(d,V),Xtr,tr_y,Xva,va_y,'next',LINEAR_EPOCHS,LINEAR_LR); nm=eval_next(npb,Xte,te_y)
        rows.append({**asdict(c),"probe_type":"linear","layer_index":li,"layer_fraction":li/model.config.num_hidden_layers,"hidden_dim":d,"H_star":Hs,"H_L":HL,"info_gap":HL-Hs,**sm,**nm,"next_excess_over_HL":nm["next_ce"]-HL})
        del sp,npb; clean_gpu()
    if RUN_FINAL_MLP_CONTROL:
        li=max(layers); Xtr,Xva,Xte=tr[li],va[li],te[li]; d=Xtr.shape[1]
        sp=train_probe(MLPState(d,c.k,V),Xtr,tr_s,Xva,va_s,'state',MLP_EPOCHS,MLP_LR); sm=eval_state(sp,Xte,te_s)
        npb=train_probe(MLPNext(d,V),Xtr,tr_y,Xva,va_y,'next',MLP_EPOCHS,MLP_LR); nm=eval_next(npb,Xte,te_y)
        rows.append({**asdict(c),"probe_type":"mlp_final","layer_index":li,"layer_fraction":1.0,"hidden_dim":d,"H_star":Hs,"H_L":HL,"info_gap":HL-Hs,**sm,**nm,"next_excess_over_HL":nm["next_ce"]-HL})
        del sp,npb; clean_gpu()
    df=pd.DataFrame(rows); df.to_csv(outcsv,index=False); save_json(rd/"config.json",{**asdict(c),"V":V,"layers":layers,"mapping":mapping.tolist(),"N_TRAIN":N_TRAIN,"N_VAL":N_VAL,"N_TEST":N_TEST}); done.write_text("completed\n")
    del tr,va,te,P; clean_gpu(); return df


## Sanity checks
The full run will not start until both model architectures pass shape, mapping, forward-pass, and `L=k` state-alignment assertions.

In [ ]:

def sanity(model_key,model_id):
    m=load_frozen(model_id); mapping=choose_token_ids(m,V,123); assert len(set(mapping.tolist()))==V and mapping.max()<m.config.vocab_size
    P=generate_kernel(V,3,123); st,ob,_=generate_dataset(P,V,3,3,32,777); assert torch.equal(st,ob)
    st2,ob2,_=generate_dataset(P,V,3,2,32,888); assert st2.shape[1]==3 and ob2.shape[1]==2
    layers=layer_indices(m); rep=extract(m,map_symbols(ob,mapping),layers,16); assert all(rep[x].shape[0]==32 for x in layers); assert all(torch.isfinite(rep[x].float()).all() for x in layers)
    print("SANITY PASS",model_key,"layers",layers,"hidden",rep[layers[-1]].shape[-1]); del m,P,rep; clean_gpu()
for mk,mid in MODELS.items(): sanity(mk,mid)
print("ALL SANITY CHECKS PASSED")


## Full experiment

In [ ]:

START=time.time(); all_rows=[]
for mk,mid in MODELS.items():
    print("="*80,"\nMODEL",mk,mid); model=load_frozen(mid)
    for k in KS:
        for L in context_lengths(k):
            for seed in SEEDS:
                c=ExpConfig(mk,mid,k,L,seed); print("RUN",c.run_id()); all_rows.append(run_config(model,c))
    del model; clean_gpu()
results=pd.concat(all_rows,ignore_index=True); results.to_csv(CSV_DIR/"part2_probe_results_raw.csv",index=False)
print("Rows",len(results),"elapsed h",round((time.time()-START)/3600,2))


## Summary tables and figures

In [ ]:

results=pd.read_csv(CSV_DIR/"part2_probe_results_raw.csv"); results["relative_context"]=results.L-results.k
summary=results.groupby(["model_key","probe_type","k","L","relative_context","layer_index","layer_fraction"],as_index=False).agg(state_exact_mean=("state_exact_acc","mean"),state_exact_std=("state_exact_acc","std"),oldest_acc_mean=("state_oldest_acc","mean"),oldest_acc_std=("state_oldest_acc","std"),mean_pos_acc=("state_mean_pos_acc","mean"),next_ce_mean=("next_ce","mean"),next_ce_std=("next_ce","std"),next_excess_HL_mean=("next_excess_over_HL","mean"),next_excess_HL_std=("next_excess_over_HL","std"),info_gap_mean=("info_gap","mean"),n=("seed","count"))
summary.to_csv(CSV_DIR/"part2_publication_summary.csv",index=False); display(summary.head(20))


In [ ]:

linear=summary[summary.probe_type=="linear"].copy()
def final_layer_rows(model_key):
    d=linear[linear.model_key==model_key]; mx=d.layer_fraction.max(); return d[d.layer_fraction==mx]
plt.figure(figsize=(8,5.5))
for mk in sorted(linear.model_key.unique()):
    d=final_layer_rows(mk); a=d.groupby("relative_context",as_index=False).agg(y=("state_exact_mean","mean"),e=("state_exact_mean","std")); plt.errorbar(a.relative_context,a.y,yerr=a.e.fillna(0),marker='o',capsize=3,label=mk)
plt.axvline(0,linestyle='--',linewidth=1); plt.xlabel('Relative context L-k'); plt.ylabel('Exact sufficient-state recovery'); plt.title('Final-layer recovery of known sufficient state'); plt.legend(); plt.tight_layout(); plt.savefig(FIG_DIR/"part2_exact_state_vs_context.png",dpi=300); plt.savefig(FIG_DIR/"part2_exact_state_vs_context.pdf"); plt.show()
plt.figure(figsize=(8,5.5))
for mk in sorted(linear.model_key.unique()):
    d=final_layer_rows(mk); a=d.groupby("relative_context",as_index=False).agg(y=("oldest_acc_mean","mean"),e=("oldest_acc_mean","std")); plt.errorbar(a.relative_context,a.y,yerr=a.e.fillna(0),marker='o',capsize=3,label=mk)
plt.axvline(0,linestyle='--',linewidth=1); plt.axhline(1/V,linestyle=':',linewidth=1); plt.xlabel('Relative context L-k'); plt.ylabel('Oldest-state-token accuracy'); plt.title('Token entering at sufficiency boundary'); plt.legend(); plt.tight_layout(); plt.savefig(FIG_DIR/"part2_oldest_token_vs_context.png",dpi=300); plt.savefig(FIG_DIR/"part2_oldest_token_vs_context.pdf"); plt.show()
plt.figure(figsize=(8,5.5))
for mk in sorted(linear.model_key.unique()):
    d=linear[(linear.model_key==mk)&(linear.relative_context==0)]; a=d.groupby("layer_fraction",as_index=False).agg(y=("state_exact_mean","mean")); plt.plot(a.layer_fraction,a.y,marker='o',label=mk)
plt.xlabel('Normalized model depth'); plt.ylabel('Exact sufficient-state recovery'); plt.title('Layerwise state recovery at L=k'); plt.legend(); plt.tight_layout(); plt.savefig(FIG_DIR/"part2_layerwise_state_recovery.png",dpi=300); plt.savefig(FIG_DIR/"part2_layerwise_state_recovery.pdf"); plt.show()
plt.figure(figsize=(8,5.5))
for mk in sorted(linear.model_key.unique()):
    d=final_layer_rows(mk); a=d.groupby("relative_context",as_index=False).agg(y=("next_excess_HL_mean","mean"),e=("next_excess_HL_mean","std")); plt.errorbar(a.relative_context,a.y,yerr=a.e.fillna(0),marker='o',capsize=3,label=mk)
plt.axvline(0,linestyle='--',linewidth=1); plt.axhline(0,linewidth=1); plt.xlabel('Relative context L-k'); plt.ylabel('Probe CE - H_L (nats)'); plt.title('Prediction gap beyond context-information floor'); plt.legend(); plt.tight_layout(); plt.savefig(FIG_DIR/"part2_next_token_excess.png",dpi=300); plt.savefig(FIG_DIR/"part2_next_token_excess.pdf"); plt.show()


## Integrity audit and export

In [ ]:

expected=len(MODELS)*sum(len(context_lengths(k)) for k in KS)*len(SEEDS); done_dirs=[p.parent for p in RUNS_DIR.rglob("DONE")]
audit={"expected_configs":expected,"done_configs":len(done_dirs),"missing_result_files":sum(not (d/"probe_results.csv").exists() for d in done_dirs)}
num=["state_ce","state_mean_pos_acc","state_oldest_acc","state_last_acc","state_exact_acc","next_ce","next_acc","next_excess_over_HL"]
audit["nonfinite_rows"]=int((~np.isfinite(results[num].to_numpy(float))).any(1).sum()); audit["duplicate_rows"]=int(results.duplicated(["model_key","k","L","seed","probe_type","layer_index"]).sum()); save_json(EXPORTS_DIR/"part2_integrity_audit.json",audit); print(audit)
output=Path.cwd()/"Part2_Sufficiency_Representation_RESULTS.zip"; allowed={".csv",".json",".png",".pdf",".txt"}
with zipfile.ZipFile(output,"w",zipfile.ZIP_DEFLATED) as z:
    for f in PROJECT_ROOT.rglob("*"):
        if f.is_file() and f.suffix.lower() in allowed and CACHE_DIR not in f.parents: z.write(f,f.relative_to(PROJECT_ROOT))
print("ZIP",output,"MB",round(output.stat().st_size/1024**2,2))


# Interpretation rules
1. Probe decodability is **not** causal usage.
2. Do not claim a representational phase transition unless all orders/seeds support it.
3. Linear failure + MLP success means nonlinear encoding, not absence of information.
4. Random token-ID remapping reduces but does not eliminate pretrained token priors.
5. A null result is publishable if it cleanly separates input sufficiency from representation sufficiency.
